# Prise en main de SQLAlchemy

Ce TP pratique SQLAlchemy sur une base SQLite. SQLite est intégré à Python (c'est le module `sqlite3`) : la base est un simple fichier, `tp_sql.db`, sans serveur à installer. Le même code fonctionne avec PostgreSQL ou MySQL : seule l'URL du moteur change.

Les exercices s'enchaînent : chaque solution définit ce que la suite utilise. Si vous êtes bloqué, exécutez la solution avant de passer à la suite.

## Connexion

### Créer un moteur

Créez un moteur (*engine*) `engine` qui se connecte à la base SQLite du fichier `tp_sql.db`, avec les sorties de débogage activées : elles affichent chaque requête SQL que SQLAlchemy envoie à la base, ce qui permettra d'observer la suite.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

L'URL d'une base SQLite est `sqlite:///` suivi du chemin du fichier (trois barres obliques pour un chemin relatif). Le fichier est créé à la première connexion.

```python
from sqlalchemy import create_engine

engine = create_engine(..., echo=...)
```

#### Solution

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///tp_sql.db", echo=True)

### Exécuter une requête textuelle

Exécutez la requête `SELECT :a + :b`, construite avec `text`, en passant les paramètres `a=2` et `b=3`, et affichez son unique résultat avec `scalar_one()`. Faites-le deux fois : dans un bloc `with engine.connect() as conn:`, puis dans un bloc `with engine.begin() as conn:`.

Dans les sorties de débogage, quelle instruction termine chacun des deux blocs ? Laquelle des deux formes utiliser pour écrire dans la base ?

**Documentation utile**

- [Paramètres d'une requête textuelle](https://docs.sqlalchemy.org/en/21/tutorial/dbapi_transactions.html#sending-parameters)
- [`Result.scalar_one`](https://docs.sqlalchemy.org/en/21/core/connections.html#sqlalchemy.engine.Result.scalar_one)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`conn.execute` prend la requête, puis un dictionnaire de paramètres (`{"a": 2, "b": 3}`). Le résultat attendu est `5`.

```python
from sqlalchemy import text

requete = text("SELECT :a + :b")

with engine.connect() as conn:
  resultat = conn.execute(requete, ...)
  print(...)

with engine.begin() as conn:
  ...
```

#### Solution

In [ ]:
from sqlalchemy import text

requete = text("SELECT :a + :b")

with engine.connect() as conn:
  print(conn.execute(requete, {"a": 2, "b": 3}).scalar_one())

with engine.begin() as conn:
  print(conn.execute(requete, {"a": 2, "b": 3}).scalar_one())

# Le bloc connect() se termine par un ROLLBACK : ce que conn.commit() n'a pas
# validé est annulé. Le bloc begin() se termine par un COMMIT, ou par un ROLLBACK
# si une exception en sort. Pour écrire dans la base, begin() valide donc tout le
# bloc, ou rien.

## SQLAlchemy Core

### Créer des tables

Avec un objet `MetaData` nommé `metadata`, décrivez deux tables (objets `Table`) :

- `membres` (variable `table_membres`) :
    - `id`, clef primaire entière ;
    - `nom`, chaîne d'au plus 100 caractères, obligatoire ;
- `adresses` (variable `table_adresses`) :
    - `id`, clef primaire entière ;
    - `rue`, `ville`, `code_postal` et `pays`, chaînes obligatoires ;
    - `id_membre`, obligatoire, clef étrangère vers la colonne `id` de `membres`.

Créez ensuite ces tables dans la base, après avoir supprimé celles qu'une exécution précédente aurait laissées.

**Documentation utile**

- [Tables et colonnes](https://docs.sqlalchemy.org/en/21/tutorial/metadata.html#setting-up-metadata-with-table-objects)
- [Contraintes](https://docs.sqlalchemy.org/en/21/tutorial/metadata.html#declaring-simple-constraints)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

« Obligatoire » se dit `nullable=False`. Une colonne qui porte une `ForeignKey("membres.id")` en prend le type : inutile de le répéter. `metadata.drop_all(engine)` puis `metadata.create_all(engine)` suppriment puis créent toutes les tables décrites, et les sorties de débogage montrent les `CREATE TABLE`.

```python
from sqlalchemy import Column, ForeignKey, Integer, MetaData, String, Table

metadata = MetaData()

table_membres = Table(
  "membres",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("nom", String(100), nullable=False),
)

table_adresses = Table(
  "adresses",
  metadata,
  ...
)

metadata.drop_all(engine)
metadata.create_all(engine)
```

#### Solution

In [ ]:
from sqlalchemy import Column, ForeignKey, Integer, MetaData, String, Table

metadata = MetaData()

table_membres = Table(
  "membres",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("nom", String(100), nullable=False),
)

table_adresses = Table(
  "adresses",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("rue", String, nullable=False),
  Column("ville", String, nullable=False),
  Column("code_postal", String, nullable=False),
  Column("pays", String, nullable=False),
  Column("id_membre", ForeignKey("membres.id"), nullable=False),
)

metadata.drop_all(engine)
metadata.create_all(engine)

### Insérer des données

Le dictionnaire `membres` associe à chaque nom de membre la liste de ses adresses. Insérez-les dans la base avec des instructions `insert`, en une seule transaction : chaque membre, puis ses adresses, qui ont besoin de l'`id` que la base vient d'attribuer au membre. Affichez ensuite le contenu des deux tables.

**Documentation utile**

- [`CursorResult.inserted_primary_key`](https://docs.sqlalchemy.org/en/21/core/connections.html#sqlalchemy.engine.CursorResult.inserted_primary_key)

In [ ]:
membres = {
  "Jean Dupont": [
    {
      "rue": "1 rue Georges Clemenceau",
      "ville": "Nantes",
      "code_postal": "44000",
      "pays": "France",
    },
    {
      "rue": "26 boulevard de la Prairie au Duc",
      "ville": "Nantes",
      "code_postal": "44200",
      "pays": "France",
    },
  ],
  "John Doe": [
    {
      "rue": "Quai Ferdinand Favre",
      "ville": "Nantes",
      "code_postal": "44000",
      "pays": "France",
    },
    {
      "rue": "Dyrehaven",
      "ville": "Klampenborg",
      "code_postal": "2930",
      "pays": "Danemark",
    },
  ],
  "Sherlock Holmes": [
    {
      "rue": "221b Baker Street",
      "ville": "Londres",
      "code_postal": "NW1 6XE",
      "pays": "Royaume-Uni",
    },
  ],
}

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`conn.execute(insert(table_membres).values(nom=nom))` insère une ligne, et `resultat.inserted_primary_key[0]` est l'`id` qu'elle a reçu. `conn.execute(insert(table_adresses), [...])` insère plusieurs lignes d'un coup, une par dictionnaire de la liste, et `dict(adresse, id_membre=id_membre)` complète un dictionnaire d'adresse. Vous devez obtenir 3 membres et 5 adresses.

```python
from sqlalchemy import insert, select

with engine.begin() as conn:
  for nom, adresses in membres.items():
    resultat = conn.execute(...)
    id_membre = ...
    conn.execute(
      insert(table_adresses),
      [dict(adresse, id_membre=id_membre) for adresse in adresses],
    )

with engine.connect() as conn:
  for ligne in conn.execute(select(table_membres)):
    print(ligne)
  ...
```

#### Solution

In [ ]:
from sqlalchemy import insert, select

with engine.begin() as conn:
  for nom, adresses in membres.items():
    resultat = conn.execute(insert(table_membres).values(nom=nom))
    id_membre = resultat.inserted_primary_key[0]
    conn.execute(
      insert(table_adresses),
      [dict(adresse, id_membre=id_membre) for adresse in adresses],
    )

with engine.connect() as conn:
  for ligne in conn.execute(select(table_membres)):
    print(ligne)
  for ligne in conn.execute(select(table_adresses)):
    print(ligne)

### Annuler une transaction

Dans un bloc `with engine.begin() as conn:`, insérez un membre `Arsène Lupin`, puis une adresse de ce membre sans `pays` : la base la refuse, puisque `pays` est obligatoire. Rattrapez l'exception `IntegrityError` en dehors du bloc, puis comptez les membres nommés `Arsène Lupin`. Qu'est devenu le membre inséré avant l'erreur ?

**Documentation utile**

- [`IntegrityError`](https://docs.sqlalchemy.org/en/21/core/exceptions.html#sqlalchemy.exc.IntegrityError)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`IntegrityError` s'importe de `sqlalchemy.exc`. Quand une exception sort d'un bloc `begin()`, SQLAlchemy annule la transaction (`ROLLBACK` dans les sorties de débogage) au lieu de la valider.

```python
from sqlalchemy.exc import IntegrityError

try:
  with engine.begin() as conn:
    resultat = conn.execute(insert(table_membres).values(nom="Arsène Lupin"))
    conn.execute(insert(table_adresses).values(...))
except IntegrityError as erreur:
  print("Refusé :", erreur.orig)

with engine.connect() as conn:
  lupins = conn.scalars(select(table_membres.c.id).where(...)).all()
  print(len(lupins))
```

#### Solution

In [ ]:
from sqlalchemy.exc import IntegrityError

try:
  with engine.begin() as conn:
    resultat = conn.execute(insert(table_membres).values(nom="Arsène Lupin"))
    conn.execute(
      insert(table_adresses).values(
        rue="36 rue Raynouard",
        ville="Paris",
        code_postal="75016",
        id_membre=resultat.inserted_primary_key[0],
      )
    )
except IntegrityError as erreur:
  print("Refusé :", erreur.orig)

with engine.connect() as conn:
  lupins = conn.scalars(
    select(table_membres.c.id).where(table_membres.c.nom == "Arsène Lupin")
  ).all()
  print(len(lupins))

# 0 : l'adresse refusée a fait sortir une exception du bloc begin(), qui a donc
# annulé toute la transaction (ROLLBACK), insertion du membre comprise.

### Sélectionner des données

Affichez, triés par ordre alphabétique, les noms des membres qui ont au moins une adresse en France, chacun une seule fois.

**Documentation utile**

- [Jointures](https://docs.sqlalchemy.org/en/21/tutorial/data_select.html#explicit-from-clauses-and-joins)
- [`Select.distinct`](https://docs.sqlalchemy.org/en/21/core/selectable.html#sqlalchemy.sql.expression.Select.distinct)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`select(table_membres.c.nom).join(table_adresses)` joint les deux tables : SQLAlchemy trouve la condition de jointure grâce à la clef étrangère. Un membre apparaît une fois par adresse en France, d'où `distinct()`. `conn.scalars(...)` renvoie la première colonne de chaque ligne. Vous devez obtenir `['Jean Dupont', 'John Doe']`.

```python
with engine.connect() as conn:
  noms = conn.scalars(
    select(table_membres.c.nom)
    .join(...)
    .where(...)
    .distinct()
    .order_by(table_membres.c.nom)
  ).all()
  print(noms)
```

#### Solution

In [ ]:
with engine.connect() as conn:
  noms = conn.scalars(
    select(table_membres.c.nom)
    .join(table_adresses)
    .where(table_adresses.c.pays == "France")
    .distinct()
    .order_by(table_membres.c.nom)
  ).all()
  print(noms)

### Mettre à jour des données

Avec une instruction `update`, renommez `John Doe` en `Jane Doe`. Affichez la table `membres` avant et après la mise à jour.

**Documentation utile**

- [`update`](https://docs.sqlalchemy.org/en/21/tutorial/data_update.html#the-update-sql-expression-construct)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`update(table).where(condition).values(colonne=valeur)` : faites-la dans un bloc `begin()`, pour qu'elle soit validée.

```python
from sqlalchemy import update

with engine.connect() as conn:
  print(conn.execute(select(table_membres)).all())

with engine.begin() as conn:
  conn.execute(update(table_membres).where(...).values(...))

with engine.connect() as conn:
  print(...)
```

#### Solution

In [ ]:
from sqlalchemy import update

with engine.connect() as conn:
  print(conn.execute(select(table_membres)).all())

with engine.begin() as conn:
  conn.execute(
    update(table_membres)
    .where(table_membres.c.nom == "John Doe")
    .values(nom="Jane Doe")
  )

with engine.connect() as conn:
  print(conn.execute(select(table_membres)).all())

### Supprimer des données

Avec des instructions `delete`, supprimez `Jane Doe` en une seule transaction, puis affichez le nombre d'adresses restantes. Supprimez d'abord ses adresses : sinon, elles désigneraient un membre qui n'existe plus. PostgreSQL refuserait alors de supprimer le membre ; SQLite ne vérifie les clefs étrangères que si on le lui demande.

**Documentation utile**

- [`delete`](https://docs.sqlalchemy.org/en/21/tutorial/data_update.html#the-delete-sql-expression-construct)
- [`Select.scalar_subquery`](https://docs.sqlalchemy.org/en/21/core/selectable.html#sqlalchemy.sql.expression.Select.scalar_subquery)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`select(table_membres.c.id).where(...)` donne l'`id` de Jane Doe, et `.scalar_subquery()` en fait une sous-requête qu'on peut comparer à une colonne dans la condition du `delete` des adresses. Il doit rester 3 adresses.

```python
from sqlalchemy import delete

id_jane = select(table_membres.c.id).where(...).scalar_subquery()

with engine.begin() as conn:
  conn.execute(delete(table_adresses).where(table_adresses.c.id_membre == id_jane))
  conn.execute(delete(table_membres).where(...))

with engine.connect() as conn:
  print(len(conn.execute(select(table_adresses)).all()))
```

#### Solution

In [ ]:
from sqlalchemy import delete

id_jane = (
  select(table_membres.c.id).where(table_membres.c.nom == "Jane Doe").scalar_subquery()
)

with engine.begin() as conn:
  conn.execute(delete(table_adresses).where(table_adresses.c.id_membre == id_jane))
  conn.execute(delete(table_membres).where(table_membres.c.nom == "Jane Doe"))

with engine.connect() as conn:
  print(len(conn.execute(select(table_adresses)).all()))

## SQLAlchemy ORM

On reprend les mêmes données avec l'ORM : des classes Python décrivent les tables, et une session synchronise leurs objets avec la base. Les tables `membres` et `adresses` sont recréées.

### Décrire les modèles

Décrivez les deux tables par des classes, en style typé : une classe `Base` qui hérite de `DeclarativeBase`, puis :

- `Membre` (table `membres`) : `id`, clef primaire entière ; `nom`, chaîne d'au plus 100 caractères ; `adresses`, la liste de ses adresses ;
- `Adresse` (table `adresses`) : `id`, clef primaire entière ; `rue`, `ville`, `code_postal` et `pays`, chaînes ; `id_membre`, clef étrangère vers `membres.id` ; `membre`, le membre auquel elle appartient.

Toutes les colonnes sont obligatoires. `Membre.adresses` et `Adresse.membre` sont les deux côtés d'une même relation un-à-plusieurs. Une adresse n'existe pas sans son membre : déclarez sur `Membre.adresses` la cascade `"all, delete-orphan"`, qui supprime les adresses d'un membre supprimé, ou retirées de sa liste.

Supprimez les tables existantes, puis créez celles-ci.

**Documentation utile**

- [Relation un-à-plusieurs](https://docs.sqlalchemy.org/en/21/orm/basic_relationships.html#one-to-many)
- [Cascade `delete-orphan`](https://docs.sqlalchemy.org/en/21/orm/cascades.html#delete-orphan)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Avec `Mapped[...]`, l'annotation donne le type et l'obligation : `Mapped[str]` est une colonne obligatoire (`Mapped[str | None]` la rendrait facultative), et `mapped_column` ne sert que pour le reste (clef primaire, longueur, clef étrangère). Chaque côté de la relation nomme l'autre avec `back_populates`. Les sorties de débogage doivent montrer deux `CREATE TABLE`, dont un avec une `FOREIGN KEY`.

```python
from sqlalchemy import ForeignKey, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
  pass


class Membre(Base):
  __tablename__ = "membres"
  id: Mapped[int] = mapped_column(primary_key=True)
  nom: Mapped[str] = mapped_column(String(100))
  adresses: Mapped[list["Adresse"]] = relationship(
    back_populates=..., cascade=...
  )


class Adresse(Base):
  __tablename__ = "adresses"
  id: Mapped[int] = mapped_column(primary_key=True)
  rue: Mapped[str]
  ...
  id_membre: Mapped[int] = mapped_column(...)
  membre: Mapped[Membre] = relationship(...)


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
```

#### Solution

In [ ]:
from sqlalchemy import ForeignKey, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
  pass


class Membre(Base):
  __tablename__ = "membres"
  id: Mapped[int] = mapped_column(primary_key=True)
  nom: Mapped[str] = mapped_column(String(100))
  adresses: Mapped[list["Adresse"]] = relationship(
    back_populates="membre", cascade="all, delete-orphan"
  )


class Adresse(Base):
  __tablename__ = "adresses"
  id: Mapped[int] = mapped_column(primary_key=True)
  rue: Mapped[str]
  ville: Mapped[str]
  code_postal: Mapped[str]
  pays: Mapped[str]
  id_membre: Mapped[int] = mapped_column(ForeignKey("membres.id"))
  membre: Mapped[Membre] = relationship(back_populates="adresses")


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

### Insérer des objets

Avec une `Session`, insérez les membres du dictionnaire `membres` et leurs adresses : créez des objets `Membre` et `Adresse`, ajoutez-les à la session, puis validez. Les sorties de débogage montrent les `INSERT` que la session envoie au `commit()`.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Il suffit d'ajouter les membres : leurs adresses, passées par `adresses=[...]`, suivent par la relation. `Adresse(**adresse)` crée une adresse à partir d'un dictionnaire de `membres`.

```python
from sqlalchemy.orm import Session

with Session(engine) as session:
  for nom, adresses in membres.items():
    membre = Membre(nom=nom, adresses=[...])
    session.add(membre)
  session.commit()
```

#### Solution

In [ ]:
from sqlalchemy.orm import Session

with Session(engine) as session:
  for nom, adresses in membres.items():
    membre = Membre(nom=nom, adresses=[Adresse(**adresse) for adresse in adresses])
    session.add(membre)
  session.commit()

### Sélectionner des objets

Affichez, triés par ordre alphabétique, les noms des membres qui ont au moins une adresse en France, avec une requête `select` sur `Membre` exécutée par une session.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`select(Membre).join(Membre.adresses)` joint par la relation, `.where(Adresse.pays == "France")` filtre, `.distinct()` évite les doublons et `.order_by(Membre.nom)` trie. `session.scalars(...)` renvoie des objets `Membre`. Vous devez obtenir Jean Dupont et John Doe.

```python
with Session(engine) as session:
  requete = select(Membre).join(...).where(...).distinct()
  for membre in session.scalars(requete):
    print(membre.nom)
```

#### Solution

In [ ]:
with Session(engine) as session:
  requete = (
    select(Membre)
    .join(Membre.adresses)
    .where(Adresse.pays == "France")
    .distinct()
    .order_by(Membre.nom)
  )
  for membre in session.scalars(requete):
    print(membre.nom)

### Parcourir une relation

Affichez chaque membre, puis la ville et le pays de chacune de ses adresses, en parcourant `membre.adresses`. Combien de `SELECT` les sorties de débogage montrent-elles ? Recommencez en ajoutant `.options(selectinload(Membre.adresses))` à la requête : combien de `SELECT` maintenant ?

**Documentation utile**

- [Stratégies de chargement](https://docs.sqlalchemy.org/en/21/tutorial/orm_related_objects.html#loader-strategies)
- [`selectinload`](https://docs.sqlalchemy.org/en/21/orm/queryguide/relationships.html#sqlalchemy.orm.selectinload)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Par défaut, une relation est chargée à la demande (*lazy loading*) : le premier accès à `membre.adresses` envoie une requête pour ce seul membre. `selectinload` charge les adresses de tous les membres en une seule requête de plus.

```python
from sqlalchemy.orm import selectinload

with Session(engine) as session:
  for membre in session.scalars(select(Membre)):
    print(membre.nom)
    for adresse in membre.adresses:
      print("  ", adresse.ville, adresse.pays)

with Session(engine) as session:
  requete = select(Membre).options(...)
  ...
```

#### Solution

In [ ]:
from sqlalchemy.orm import selectinload

with Session(engine) as session:
  for membre in session.scalars(select(Membre)):
    print(membre.nom)
    for adresse in membre.adresses:
      print("  ", adresse.ville, adresse.pays)

print("*" * 50)

with Session(engine) as session:
  requete = select(Membre).options(selectinload(Membre.adresses))
  for membre in session.scalars(requete):
    print(membre.nom)
    for adresse in membre.adresses:
      print("  ", adresse.ville, adresse.pays)

# Sans option : 4 SELECT, un pour les membres, puis un par membre au premier
# accès à membre.adresses (chargement paresseux). Avec selectinload : 2 SELECT,
# un pour les membres, un pour les adresses de tous les membres (WHERE ... IN).
# Sur des milliers de membres, l'écart compte : c'est le problème « N + 1 ».

### Modifier un objet

Récupérez le membre `John Doe` avec une requête `select`, renommez-le `Jane Doe` en modifiant son attribut `nom`, puis validez. Affichez la liste des noms des membres avant et après.

**Documentation utile**

- [Modifier un objet](https://docs.sqlalchemy.org/en/21/tutorial/orm_data_manipulation.html#updating-orm-objects-using-the-unit-of-work-pattern)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`session.scalars(requete).one()` renvoie l'unique objet trouvé, et lève une exception s'il y en a zéro ou plusieurs. Au `commit()`, la session envoie elle-même l'`UPDATE` : les sorties de débogage le montrent.

```python
with Session(engine) as session:
  print(session.scalars(select(Membre.nom)).all())
  john = session.scalars(select(Membre).where(...)).one()
  john.nom = ...
  session.commit()
  print(...)
```

#### Solution

In [ ]:
with Session(engine) as session:
  print(session.scalars(select(Membre.nom)).all())
  john = session.scalars(select(Membre).where(Membre.nom == "John Doe")).one()
  john.nom = "Jane Doe"
  session.commit()
  print(session.scalars(select(Membre.nom)).all())

### Supprimer un objet

Récupérez `Jane Doe`, supprimez-la avec la session, validez, puis affichez le nombre d'adresses restantes dans la base. Qu'est-il arrivé à ses adresses ? Qu'en aurait-il été sans la cascade `"all, delete-orphan"` de `Membre.adresses` ?

**Documentation utile**

- [Supprimer un objet](https://docs.sqlalchemy.org/en/21/tutorial/orm_data_manipulation.html#deleting-orm-objects-using-the-unit-of-work-pattern)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`session.delete(objet)` marque l'objet à supprimer, et le `commit()` envoie les `DELETE` : regardez lesquels dans les sorties de débogage. Il doit rester 3 adresses.

```python
with Session(engine) as session:
  jane = session.scalars(select(Membre).where(...)).one()
  session.delete(...)
  session.commit()
  print(len(session.scalars(select(Adresse)).all()))
```

#### Solution

In [ ]:
with Session(engine) as session:
  jane = session.scalars(select(Membre).where(Membre.nom == "Jane Doe")).one()
  session.delete(jane)
  session.commit()
  print(len(session.scalars(select(Adresse)).all()))

# 3 adresses : la cascade « all, delete-orphan » a supprimé les deux adresses de
# Jane Doe avec elle (un DELETE des adresses, puis celui du membre). Sans cascade,
# SQLAlchemy aurait mis leur id_membre à NULL (un UPDATE), ce que la colonne
# obligatoire refuse : le commit() aurait levé une IntegrityError.